# YOLO26-M (Ultralytics) — evaluation

Scores the checkpoint written by [`yolo26m_train.ipynb`](yolo26m_train.ipynb) (`models/yolo26m/weights/best.pt`) on the test split.

The best checkpoint (by Ultralytics' validation fitness) is run on the test images, resized to 640×640 (no augmentation), and scored with
the `EvaluationFramework`, exactly like the torchvision models. `model.predict` gets BGR arrays at 640×640 with
`conf=0.001` and `max_det=100`; latency is the sum of Ultralytics' pre-process, inference and post-process times.

In [ ]:
import numpy as np
from ultralytics import YOLO

from ppe_detection_evaluation import YOLO26_M, EvaluationFramework
from ppe_detection_evaluation.dataset.augmentations import IMAGE_SIZE
from ppe_detection_evaluation.dataset.core import check_dataset, list_images, load_eval_example, xyxy_to_cxcywh
from ppe_detection_evaluation.experiment import DATA_ROOT, MODELS_DIR, RESULTS_DIR

check_dataset(DATA_ROOT)  # the training notebook downloads the dataset

In [ ]:
checkpoint = MODELS_DIR / "yolo26m" / "weights" / "best.pt"
model = YOLO(checkpoint)

In [ ]:
fw = EvaluationFramework(YOLO26_M)
for path in list_images(DATA_ROOT, "test"):
    image, gt_boxes, gt_labels = load_eval_example(path)
    result = model.predict(
        np.ascontiguousarray(image[..., ::-1]),  # predict() expects BGR arrays
        imgsz=IMAGE_SIZE,
        conf=fw.score_threshold,                 # 0.001: keep low-confidence detections for the AP / PR-curve metrics
        max_det=fw.max_detections,
        verbose=False,
    )[0]
    fw.add(
        path.stem,
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.cls.cpu().numpy(),
        xyxy_to_cxcywh(gt_boxes, IMAGE_SIZE, IMAGE_SIZE),
        gt_labels,
        latency_ms=sum(result.speed.values()),   # Ultralytics' pre-process + inference + post-process time
    )

results = fw.evaluate()
fw.save(RESULTS_DIR / "yolo26m_test.json")

In [ ]:
det = results["detection"]["groups"]
print(f"mAP@[.5:.95] {det['all']['mAP']:.4f}   mAP@.5 {det['all']['mAP50']:.4f}")
print(f"PPE mAP@.5 {det['ppe']['mAP50']:.4f}   small PPE mAP@.5 {results['detection']['by_size']['small']['groups']['ppe']['mAP50']:.4f}")
print("violation mean:", results["violation"]["mean"])
print("latency:", results["latency"])